# Gaussian baseline：Random Forest 与 HGBR 的 DML 对比（$p=5$）

本 notebook 只使用 $U\sim N(0,1)$ baseline，在完全相同的数据、随机种子和 5 折 cross-fitting 划分下比较两类 nuisance learner：

- `HGBR`：原代码的 `HistGradientBoostingRegressor`；
- `RF`：`RandomForestRegressor`。

两类学习器都分别进入 Mean DML 和 Median DML。这样可以判断原 Gaussian coverage 偏低是否主要与 HGBR 的第一阶段拟合有关。

In [ ]:
# ============================================================
# 调参区
# ============================================================
P = 5
N = 1000
THETA_0 = 1.0
NORMAL_BASELINE = "N(0,1)"
DGP_NAMES = ["DGP1_linear", "DGP2_nonlinear"]
LEARNERS = ["HGBR", "RF"]

N_FOLDS = 5
N_REPLICATIONS = 1000
BASE_SEED = 20260826
N_JOBS = -1              # Monte Carlo 外层并行
RF_N_JOBS = 1            # 避免与外层并行发生 CPU 过度占用

QUICK_TEST = False
QUICK_N = 300
QUICK_REPLICATIONS = 2
QUICK_RF_TREES = 20

HGBR_PARAMS = {
    "learning_rate": 0.05,
    "max_iter": 100,
    "max_leaf_nodes": 15,
    "min_samples_leaf": 20,
    "l2_regularization": 1.0,
}

RF_PARAMS = {
    "n_estimators": 300,
    "max_depth": None,
    "min_samples_leaf": 5,
    "max_features": 1.0,
    "bootstrap": True,
}

SAVE_RESULTS = True
OUTPUT_CSV = None  # 导入依赖后自动设为桌面 CSV

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
from scipy.stats import norm
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore", category=RuntimeWarning)
sns.set_theme(style="whitegrid", context="notebook")

if OUTPUT_CSV is None:
    OUTPUT_CSV = Path.home() / "Desktop" / "dml_p5_gaussian_RF_vs_HGBR_summary.csv"
else:
    OUTPUT_CSV = Path(OUTPUT_CSV)

assert P == 5
assert set(LEARNERS) == {"HGBR", "RF"}
assert N_FOLDS >= 2
print(f"参数检查完成；正式结果将保存到：{OUTPUT_CSV}")

## 1. 原始 DGP 与 Gaussian baseline

数据生成与原 `p=5` Student-t notebook 的 Gaussian baseline 完全一致。两种学习器在每次重复中使用同一份数据，并使用同一个 KFold 划分，从而形成配对比较。

In [ ]:
def ar1_covariance(p: int, rho: float = 0.5):
    index = np.arange(p)
    return rho ** np.abs(index[:, None] - index[None, :])


def nuisance_functions(X: np.ndarray, dgp_name: str):
    x1, x2, x3, x4, x5 = X.T
    if dgp_name == "DGP1_linear":
        m0 = 0.80*x1 - 0.50*x2 + 0.30*x3 + 0.20*x4 - 0.10*x5
        g0 = 1.00*x1 + 0.50*x2 - 0.50*x3 + 0.25*x4 + 0.10*x5
    elif dgp_name == "DGP2_nonlinear":
        m0 = (0.60*np.sin(x1) + 0.35*(x2**2 - 1.0)
              + 0.25*x3*x4 - 0.20*np.tanh(x5))
        g0 = (1.20*np.sin(x1) + 0.70*(x2**2 - 1.0)
              + 0.50*x3*x4 + 0.30*np.tanh(x5))
    else:
        raise ValueError(f"未知 DGP: {dgp_name}")
    return m0, g0


def generate_data(n: int, dgp_name: str, seed: int):
    rng = np.random.default_rng(seed)
    X = rng.multivariate_normal(np.zeros(P), ar1_covariance(P), size=n)
    m0, g0 = nuisance_functions(X, dgp_name)
    V = rng.normal(0.0, 1.0, size=n)
    U = rng.normal(0.0, 1.0, size=n)
    D = m0 + V
    Y = THETA_0*D + g0 + U
    return X, D, Y, U


X_demo, D_demo, Y_demo, U_demo = generate_data(N, DGP_NAMES[0], BASE_SEED)
print(f"X shape={X_demo.shape}; U mean={U_demo.mean():.3f}; U variance={U_demo.var(ddof=1):.3f}")

## 2. HGBR 与 RF nuisance learners

- Mean DML：HGBR 使用平方误差；RF 使用 `criterion="squared_error"`。
- Median DML：HGBR 使用原生 0.5 分位数损失；RF 使用 `criterion="absolute_error"`，使各回归树采用 L1/叶节点中位数思想。需要注意，随机森林最终会平均各棵树的预测，因此它是面向中位数的 RF 近似，并不是严格的 Quantile Regression Forest。

RF 的绝对误差准则通常明显慢于平方误差准则。正式运行 1000 次重复可能耗时较长。

In [ ]:
def make_nuisance_model(learner: str, kind: str, seed: int):
    if learner == "HGBR":
        params = dict(HGBR_PARAMS, random_state=seed)
        if kind == "mean":
            return HistGradientBoostingRegressor(loss="squared_error", **params)
        if kind == "median":
            return HistGradientBoostingRegressor(loss="quantile", quantile=0.5, **params)

    if learner == "RF":
        params = dict(RF_PARAMS)
        params["n_estimators"] = QUICK_RF_TREES if QUICK_TEST else RF_PARAMS["n_estimators"]
        params.update(random_state=seed, n_jobs=RF_N_JOBS)
        if kind == "mean":
            return RandomForestRegressor(criterion="squared_error", **params)
        if kind == "median":
            return RandomForestRegressor(criterion="absolute_error", **params)

    raise ValueError("learner 必须为 HGBR/RF，kind 必须为 mean/median。")


def cross_fitted_residuals(X, D, Y, learner: str, kind: str, seed: int):
    d_hat = np.empty_like(D, dtype=float)
    y_hat = np.empty_like(Y, dtype=float)
    splitter = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (train_idx, test_idx) in enumerate(splitter.split(X)):
        model_d = make_nuisance_model(learner, kind, seed + 100*fold + 1)
        model_y = make_nuisance_model(learner, kind, seed + 100*fold + 2)
        model_d.fit(X[train_idx], D[train_idx])
        model_y.fit(X[train_idx], Y[train_idx])
        d_hat[test_idx] = model_d.predict(X[test_idx])
        y_hat[test_idx] = model_y.predict(X[test_idx])
    return D - d_hat, Y - y_hat


def weighted_median(values: np.ndarray, weights: np.ndarray):
    order = np.argsort(values)
    values_sorted = values[order]
    weights_sorted = weights[order]
    cutoff = 0.5*weights_sorted.sum()
    index = np.searchsorted(np.cumsum(weights_sorted), cutoff, side="left")
    return float(values_sorted[min(index, len(values_sorted)-1)])


def fit_mean_dml(d_res, y_res):
    jacobian = np.mean(d_res**2)
    if jacobian <= 1e-12:
        return np.nan, np.nan
    theta_hat = np.mean(d_res*y_res) / jacobian
    score = d_res * (y_res - theta_hat*d_res)
    se_hat = np.sqrt(np.var(score/jacobian, ddof=1) / len(d_res))
    return float(theta_hat), float(se_hat)


def fit_median_dml(d_res, y_res):
    keep = np.abs(d_res) > 1e-10
    if not np.any(keep):
        return np.nan, np.nan
    theta_hat = weighted_median(y_res[keep]/d_res[keep], np.abs(d_res[keep]))
    residual = y_res - theta_hat*d_res
    n = len(residual)
    mad = np.median(np.abs(residual - np.median(residual))) / 0.67448975
    robust_scale = max(float(mad), 1e-6)
    bandwidth = max(1.06*robust_scale*n**(-1.0/5.0), 1e-6)
    kernel_at_zero = norm.pdf(residual/bandwidth) / bandwidth
    A_hat = np.mean((d_res**2)*kernel_at_zero)
    B_hat = 0.25*np.mean(d_res**2)
    se_hat = np.sqrt(B_hat/(A_hat**2*n)) if A_hat > 1e-12 else np.nan
    return float(theta_hat), float(se_hat)


def estimate_all_methods(X, D, Y, seed: int):
    estimates = []
    for learner in LEARNERS:
        d_mean, y_mean = cross_fitted_residuals(X, D, Y, learner, "mean", seed)
        estimates.append((learner, "Mean_DML", *fit_mean_dml(d_mean, y_mean)))
        d_median, y_median = cross_fitted_residuals(X, D, Y, learner, "median", seed)
        estimates.append((learner, "Median_DML", *fit_median_dml(d_median, y_median)))
    return estimates

In [ ]:
def one_replication(dgp_name: str, replication: int, n_use: int):
    dgp_id = DGP_NAMES.index(dgp_name)
    seed = BASE_SEED + 100_000*dgp_id + replication
    X, D, Y, U = generate_data(n_use, dgp_name, seed)
    estimates = estimate_all_methods(X, D, Y, seed + 500_000)
    rows = []
    for learner, estimator, theta_hat, se_hat in estimates:
        ci_low = theta_hat - 1.96*se_hat
        ci_high = theta_hat + 1.96*se_hat
        rows.append({
            "dgp": dgp_name, "p": P, "n": n_use,
            "error_distribution": NORMAL_BASELINE,
            "learner": learner, "estimator": estimator,
            "method": f"{learner}_{estimator}",
            "replication": replication, "theta_hat": theta_hat, "se_hat": se_hat,
            "ci_low": ci_low, "ci_high": ci_high,
            "covered": bool(ci_low <= THETA_0 <= ci_high),
            "sample_u_variance": np.var(U, ddof=1),
        })
    return rows


def run_simulation():
    n_use = QUICK_N if QUICK_TEST else N
    r_use = QUICK_REPLICATIONS if QUICK_TEST else N_REPLICATIONS
    tasks = [(dgp, rep, n_use) for dgp in DGP_NAMES for rep in range(r_use)]
    mode = "QUICK_TEST" if QUICK_TEST else "FORMAL"
    rf_trees = QUICK_RF_TREES if QUICK_TEST else RF_PARAMS["n_estimators"]
    print(f"开始 [{mode}]：p={P}, n={n_use}, repetitions={r_use}, RF trees={rf_trees}")
    start = time.perf_counter()
    blocks = Parallel(n_jobs=N_JOBS, verbose=5)(delayed(one_replication)(*task) for task in tasks)
    rows = [row for block in blocks for row in block]
    print(f"完成，用时 {time.perf_counter()-start:.1f} 秒，共 {len(rows)} 条结果。")
    return pd.DataFrame(rows)


results = run_simulation()
results.head()

In [ ]:
def summarize_simulation(results_df):
    work = results_df.copy()
    work["error"] = work["theta_hat"] - THETA_0
    work["squared_error"] = work["error"]**2
    summary = (work.groupby(
        ["dgp", "p", "error_distribution", "learner", "estimator", "method"],
        as_index=False,
    ).agg(
        mean_estimate=("theta_hat", "mean"), bias=("error", "mean"),
        sd=("theta_hat", "std"), mean_se=("se_hat", "mean"),
        mse=("squared_error", "mean"), coverage=("covered", "mean"),
        mean_sample_u_variance=("sample_u_variance", "mean"),
        valid_runs=("theta_hat", "count"),
    ))
    summary["rmse"] = np.sqrt(summary.pop("mse"))
    return summary.sort_values(["dgp", "estimator", "learner"]).reset_index(drop=True)


summary = summarize_simulation(results)
display(summary.round({
    "mean_estimate": 4, "bias": 4, "sd": 4, "mean_se": 4,
    "rmse": 4, "coverage": 3, "mean_sample_u_variance": 3,
}))

# RF-HGBR 的配对估计差，帮助判断差异是否稳定
paired = results.pivot_table(
    index=["dgp", "replication", "estimator"], columns="learner", values="theta_hat"
).reset_index()
paired["RF_minus_HGBR"] = paired["RF"] - paired["HGBR"]
paired_summary = paired.groupby(["dgp", "estimator"], as_index=False).agg(
    mean_RF_minus_HGBR=("RF_minus_HGBR", "mean"),
    sd_RF_minus_HGBR=("RF_minus_HGBR", "std"),
)
display(paired_summary.round(4))

if SAVE_RESULTS and not QUICK_TEST:
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    summary.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"正式汇总结果已保存：{OUTPUT_CSV.resolve()}")
elif QUICK_TEST:
    print("QUICK_TEST=True：不写出测试 CSV。")
else:
    print("SAVE_RESULTS=False：未写出 CSV。")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (metric, label) in zip(axes, [("bias", "Bias"), ("rmse", "RMSE"), ("coverage", "95% CI coverage")]):
    sns.barplot(data=summary, x="dgp", y=metric, hue="method", ax=ax)
    ax.set_xlabel("DGP")
    ax.set_ylabel(label)
    ax.tick_params(axis="x", rotation=20)
    if metric == "bias": ax.axhline(0.0, color="black", linewidth=1, linestyle=":")
    if metric == "coverage":
        ax.axhline(0.95, color="black", linewidth=1, linestyle=":")
        ax.set_ylim(0.0, 1.02)
plt.tight_layout()
plt.show()

## 如何判断是不是 HGBR 的问题

重点比较同一 DGP、同一估计器下 RF 与 HGBR 的 `bias`、`sd`、`mean_se` 和 `coverage`：

1. 如果 RF 的绝对 Bias 明显更小且 coverage 更接近 0.95，说明 HGBR 第一阶段拟合是重要原因。
2. 如果 RF 与 HGBR 都有方向和大小相近的 Bias，问题不太可能只来自 HGBR，应继续检查 DML 设定或样本量。
3. 如果 Bias 很小但 `mean_se < sd`，主要问题是标准误低估。
4. RF Median 使用绝对误差树的森林平均，并非严格 Quantile Regression Forest，因此 Median DML 的比较应注明这一限制。
5. 正式版本为 `QUICK_TEST=False`、1000 次重复和 300 棵 RF 树；运行时间可能明显长于原 HGBR notebook。完成后自动生成桌面 CSV：`dml_p5_gaussian_RF_vs_HGBR_summary.csv`。